- Business Problem
- Understanding the Dataset
- Data Collection
- Data Cleaning
- EDA
- Feature Engineering
- Feature Selection
- Data Preprocessing
- Train/Test Split
- Model Building
- Model Evaluate
- Error Analysis
- Model Interpretation and Tuning
- Saving the Model
- Make Predictions on Unseen Data

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

RANDOM_STATE = 42
pd.set_option('display.max_columns',None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Environment is ready')

Environment is ready


In [2]:
df = sns.load_dataset('diamonds')
print('Shape:', df.shape)

Shape: (53940, 10)


In [3]:
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   carat    53940 non-null  float64 
 1   cut      53940 non-null  category
 2   color    53940 non-null  category
 3   clarity  53940 non-null  category
 4   depth    53940 non-null  float64 
 5   table    53940 non-null  float64 
 6   price    53940 non-null  int64   
 7   x        53940 non-null  float64 
 8   y        53940 non-null  float64 
 9   z        53940 non-null  float64 
dtypes: category(3), float64(6), int64(1)
memory usage: 3.0 MB


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
carat,53940.0,0.797940,0.474011,0.2,0.40,0.70,1.04,5.01
depth,53940.0,61.749405,1.432621,43.0,61.00,61.80,62.50,79.00
table,53940.0,57.457184,2.234491,43.0,56.00,57.00,59.00,95.00
price,53940.0,3932.799722,3989.439738,326.0,950.00,2401.00,5324.25,18823.00
x,53940.0,5.731157,1.121761,0.0,4.71,5.70,6.54,10.74
y,53940.0,5.734526,1.142135,0.0,4.72,5.71,6.54,58.90
z,53940.0,3.538734,0.705699,0.0,2.91,3.53,4.04,31.80


In [6]:
df.describe(include='category').T

,count,unique,top,freq
cut,53940,5,Ideal,21551
color,53940,7,G,11292
clarity,53940,8,SI1,13065


In [7]:
min(df['x'])

0.0

In [8]:
print('Duplicate Rows :', df.duplicated().sum())

Duplicate Rows : 146


In [9]:
print('Missing values per column:\n', df.isnull().sum())

Missing values per column:
 carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64


In [10]:
zero_dims = df[(df['x']== 0) | (df['y'] == 0) | (df['z'] == 0)]
print(f"Rows with a zero dimension: {len(zero_dims)}")
zero_dims[['carat','x','y','z','price']].head()

Rows with a zero dimension: 20


,carat,x,y,z,price
2207,1.00,6.55,6.48,0.0,3142
2314,1.01,6.66,6.60,0.0,3167
4791,1.10,6.50,6.47,0.0,3696
5471,1.01,6.50,6.47,0.0,3837
10167,1.50,7.15,7.04,0.0,4731


In [11]:
df_clean = df.copy()

In [12]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Dropped {before - len(df_clean)} duplicate rows.")

Dropped 146 duplicate rows.
